# Preparation

## Package

In [1]:
import torch
import torch.nn as nn
from torchMPC.mpc import mpc
from torchMPC.mpc import util
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import pdb

for i in range(1):
    pass
print('GOGOGO!')

GOGOGO!


## Cost net (simple running cost and MLP terminal cost)
g = x^2 + u^2


F = xT * MLP(x) * x

In [2]:
class MPCCostNetwork(nn.Module):
    def __init__(self, n_state, n_ctrl):
        super().__init__()
        self.n_state = n_state
        self.n_ctrl = n_ctrl

        # 运行损失 xTQx + uTRu
        self.Q = torch.eye(n_state)
        self.R = torch.eye(n_ctrl)

        # 终端损失网络 xTFx
        # self.terminal_net = nn.Sequential(
        #     nn.Linear(n_state, 128),
        #     nn.ReLU(),
        #     nn.Linear(128, 64),
        #     nn.ReLU(),
        #     nn.Linear(64, n_state * n_state)
        # )

        self.F = nn.Parameter(torch.randn(n_state, n_state))

    def forward(self, tau, terminal=False):
        """
        Args:
            tau: [batch_size, n_state + n_ctrl], concatenated form of state and action
            t: Current timestamp
            T: Total prediction horizon
        Returns:
            cost: [batch_size] 
        """
        batch_size = tau.size(0)
        
        # 分离状态和控制
        # 如果t == T-1，ctrl将会是是零向量，表示不存在，所以不会index out of range
        state = tau[:, :self.n_state]
        ctrl = tau[:, self.n_state:]
        
        # 如果是最后一个时间步，只计算终端损失
        if terminal:
            terminal_F = self.F.repeat(batch_size, 1, 1)
            state = state.unsqueeze(1)
            terminal_cost = torch.bmm(torch.bmm(state, terminal_F), state.transpose(1, 2))
            return terminal_cost.squeeze(-1).squeeze(-1) # Return shape: (batch_size, )
        else:
            return self.running_cost(state, ctrl)
    
    def running_cost(self, state, ctrl): 
        # 简单二次型运行代价
        batch_size = state.shape[0]
        state = state.unsqueeze(1) # (batch_size, 1, n_state)
        ctrl = ctrl.unsqueeze(1)

        xTQx = torch.bmm(torch.bmm(state, self.Q.repeat(batch_size, 1, 1)), state.transpose(1, 2))
        uTRu = torch.bmm(torch.bmm(ctrl, self.R.repeat(batch_size, 1, 1)), ctrl.transpose(1, 2))
        return (xTQx + uTRu).squeeze(-1).squeeze(-1) # Return shape: (batch_size,)

In [3]:
testNet = MPCCostNetwork(3, 2)
testNet.Q = nn.Parameter(torch.tensor([[1, 0, 0], [0, 1, 0], [0, 0, 1]]).float())
tau = torch.randn(10, 5)
cost = testNet(tau)
print(cost)
cost = testNet(tau, terminal=True)
print(cost)

tensor([ 3.0945,  2.1109,  1.4061,  5.5018,  2.9295,  4.2386,  7.1995,  5.3521,
        14.2554,  6.2425], grad_fn=<SqueezeBackward1>)
tensor([-1.6815, -0.1754,  0.1372,  0.0823,  0.2937, -3.1844,  1.6258, -1.1211,
        -2.9197, -3.8442], grad_fn=<SqueezeBackward1>)


## Sys Dyna (F = x + u)

In [4]:
class DynamicsF(nn.Module):
    def forward(self, state, action):
        # Dimension of state & action: (batch_size, C)
        assert state.shape[-1] == action.shape[-1]
        return state + action

## Modify the original MPC lib
To support terminal cost.

## Sanity check

### Solve optimization problem use MPC

In [20]:
from torchMPC.mpc.env_dx import cartpole

cartpole = cartpole.CartpoleDx()
print(cartpole.upper)

100.0


In [21]:
# 下面和LQR的结果做了对比，结果一致。所以认定对mpc torch库的修改是正确的
LQR_ITER = 100
batch_size, T, mpc_T = 1, 10, 3
DTYPE = torch.float
nx, nu = 1, 1 # 用1, 1会出问题，最后得到的控制都是nan
n_sc = nx + nu
dynamics = DynamicsF()
cost = MPCCostNetwork(nx, nu)

torch.manual_seed(43)

init_state = torch.tensor([[1.]], dtype=DTYPE)
cost.F = nn.Parameter(torch.tensor([[1.5]], dtype=DTYPE))
cost.Q = torch.tensor([[2.5]], dtype=DTYPE)
u_init = None
x_now = init_state

VN_list = []
x_list = []
u_list = []
for t in range(T):
    ctrl = mpc.MPC(nx, nu, mpc_T, u_lower=-100.0, u_upper=100.0, lqr_iter=LQR_ITER, verbose=1,
                exit_unconverged=False, eps=1e-3, n_batch=batch_size, backprop=False, u_init=u_init, 
                grad_method=mpc.GradMethods.AUTO_DIFF)

    x_seq, u_seq, objs = ctrl(x_now, cost, dynamics)
    action = u_seq[0]

    VN_list.append(objs)
    x_list.append(x_now)
    u_list.append(action)

    x_now = dynamics(x_now, action)
    u_init = torch.cat((u_seq[1:], torch.zeros(1, batch_size, nu, dtype=DTYPE)), dim=0)
    # print(x_seq) # Shape: (mpc_T, batch_size, nx)
    # print(u_seq) # Shape: (mpc_T, batch_size, nu)
    # print(objs)  # Shape: (batch_size,)
    # Verbose > 0时，打印出来的log数据分别表示：
    # iLQR迭代次数、最优轨迹的平均代价、action更新的最大范数(表示action的变化量)、线搜索步长均值、QP子问题的总迭代次数

print(x_list)
print(u_list)
print(VN_list)

Initial mean(cost): 6.5000e+00
| 0 | nan | nan | 1.00e+00 | tensor([3.]) |
| 1 | nan | nan | 1.00e+00 | tensor([3.]) |
| 2 | nan | nan | 1.00e+00 | tensor([3.]) |
| 3 | nan | nan | 1.00e+00 | tensor([3.]) |
| 4 | nan | nan | 1.00e+00 | tensor([3.]) |
| 5 | nan | nan | 1.00e+00 | tensor([3.]) |
Initial mean(cost): nan
| 0 | nan | nan | 1.00e+00 | tensor([3.], grad_fn=<LQRStepFnBackward>) |
| 1 | nan | nan | 1.00e+00 | tensor([3.], grad_fn=<LQRStepFnBackward>) |
| 2 | nan | nan | 1.00e+00 | tensor([3.], grad_fn=<LQRStepFnBackward>) |
| 3 | nan | nan | 1.00e+00 | tensor([3.], grad_fn=<LQRStepFnBackward>) |
| 4 | nan | nan | 1.00e+00 | tensor([3.], grad_fn=<LQRStepFnBackward>) |
| 5 | nan | nan | 1.00e+00 | tensor([3.], grad_fn=<LQRStepFnBackward>) |
Initial mean(cost): nan
| 0 | nan | nan | 1.00e+00 | tensor([3.], grad_fn=<LQRStepFnBackward>) |
| 1 | nan | nan | 1.00e+00 | tensor([3.], grad_fn=<LQRStepFnBackward>) |
| 2 | nan | nan | 1.00e+00 | tensor([3.], grad_fn=<LQRStepFnBackward>) |


### Verify the RDP inequality

In [6]:
def cal_RDP_criteria(VN_list, x_list, u_list, cost_nn, alpha, MPC_T, func, eps, test=False):
    """
    Input:
        VN_list: list of the cost function value
        x_list: list of the state
        u_list: list of the control
        cost_nn: the cost function
        alpha: the weight of the running cost term
        func: the function used to incoporate the RDP inequality into the loss function
        test: whether to print the RDP value
    Return:
        RDP criteria
    """
    loss = 0
    for i in range(MPC_T - 1):
        tau = torch.cat((x_list[i], u_list[i]), dim=1)
        RDP = (VN_list[i+1] + alpha * cost_nn(tau, False)) - VN_list[i] - eps # Wish RDP <= 0
        if test:
            print(f'RDP{i}: {RDP}')
        loss += func(RDP)
    return loss

In [7]:
# 测测上面的小例子
alpha = 1
loss = cal_RDP_criteria(VN_list, x_list, u_list, cost, alpha, T, lambda x: torch.relu(x), 0, test=True)
print(loss)

RDP0: tensor([0.0093], grad_fn=<SubBackward0>)
RDP1: tensor([0.0006], grad_fn=<SubBackward0>)
RDP2: tensor([3.2863e-05], grad_fn=<SubBackward0>)
RDP3: tensor([1.9549e-06], grad_fn=<SubBackward0>)
RDP4: tensor([1.1629e-07], grad_fn=<SubBackward0>)
RDP5: tensor([6.9181e-09], grad_fn=<SubBackward0>)
RDP6: tensor([4.1155e-10], grad_fn=<SubBackward0>)
RDP7: tensor([2.4482e-11], grad_fn=<SubBackward0>)
RDP8: tensor([1.4564e-12], grad_fn=<SubBackward0>)
tensor([0.0099], grad_fn=<AddBackward0>)


## Test utils

In [8]:
from torchMPC.mpc.dynamics import AffineDynamics
from torch.optim import Adam

In [13]:
def solve_mpc_bounded(nx, nu, u_lower, u_upper, batch_size, mpc_T, ocp_T, lqr_iter, dynamics, cost, DTYPE):
    u_init = None
    x_init = torch.randn(batch_size, nx, dtype=DTYPE) * 100
    
    x_list = []
    u_list = []
    VN_list = []
    
    x_now = x_init
    for i in range(mpc_T):
        ctrl = mpc.MPC(nx, nu, ocp_T, u_lower=u_lower, u_upper=u_upper, lqr_iter=lqr_iter, verbose=0,
                    exit_unconverged=False, eps=1e-2, n_batch=batch_size, backprop=True, u_init=u_init, 
                    grad_method=mpc.GradMethods.AUTO_DIFF)
        
        x_seq, u_seq, objs = ctrl(x_now, cost, dynamics)
        action = u_seq[0]

        VN_list.append(objs)
        x_list.append(x_now)
        u_list.append(action)

        x_now = dynamics(x_now, action)
        u_init = torch.cat((u_seq[1:], torch.zeros(1, batch_size, nu, dtype=DTYPE)), dim=0)
    return x_list, u_list, VN_list

In [14]:
# Used for large scale test
def myLogger(date, msg, *args):
    try:
        # 把data和args拼接成一个字符串
        log_name = f"{date}_" + "_".join(map(str, args)) + ".txt"
        with open('./testLog/' + log_name, 'a') as file:
            file.write(msg + "\n")
    except IOError as e:
        print(f"写入文件失败: {e}")

In [15]:
# Used for hyper-parameter test and two-trainable test
def hyperLogger(dir, msg, filename):
    try:
        # 把data和args拼接成一个字符串
        with open(dir + filename + '.txt', 'a') as file:
            file.write(msg + "\n")
    except IOError as e:
        print(f"写入文件失败: {e}")